In [19]:
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from copy import deepcopy
from tqdm import tqdm
from gen_variable_standard_static import metrics_search_for_two_fragments_df,\
    find_aggregate_variable_names_gen_mod, find_all_variable_names_gen_mod,\
    metadata_part_name, metadata_agg_name, check_variable_data_exists_single_system, \
    sources_checker

In [2]:
systems_cleaned = pd.read_csv('../../../data/core/systems_cleaned.csv')
systems_cleaned.columns

Index(['system_id', 'system_public_name', 'site_location',
       'timezone_or_utc_offset', 'latitude', 'longitude', 'elevation_m',
       'dc_capacity_kW', 'kg_climate', 'pvcz_composite', 'pvcz_t_rack',
       'pvcz_t_roof', 'pvcz_humidity', 'pvcz_wind', 'tracking', 'type',
       'azimuth', 'tilt', 'first_timestamp', 'last_timestamp', 'years',
       'number_records', 'dataset_size_mb', 'available_sensor_channels',
       'qa_status', 'qa_issue', 'first_year', 'is_prize_data',
       'is_lake_parquet_data', 'is_lake_csv_data', 'has_irradiance_data',
       'has_ambient_temperature_data', 'has_temperature_data',
       'has_power_data', 'has_current_data', 'has_voltage_data', 'has_ac_data',
       'has_dc_data', 'module_type', 'simplified_type', 'system_source',
       'num_days_actual_records'],
      dtype='str')

In [3]:
metrics_dir = Path("../../../data/raw/parquet-metrics/")
metrics_pq = pq.ParquetDataset(metrics_dir)
metrics_df = metrics_pq.read().to_pandas()
metrics_id_set = set(metrics_df.system_id)

## Explore

In [4]:
metrics_search_for_two_fragments_df(
    metrics_df, 'pow', 'dc', 'and'
)

,system_id,metric_id,sensor_name,common_name,raw_units,units,calc_scale,calc_offset,calc_details,aggregation_type,source_type,source_id,comments,standard_name
0,10,422,dc_power,DC power,W,W,1.0,0.0,,avg,NaN,NaN,,dc_power__422
15,1199,2715,dc_power,DC power,W,W,1.0,0.0,inv1_dc_power+inv2_dc_power+inv3_dc_power+inv4...,avg,NaN,NaN,,dc_power__2715
17,1199,2717,inv1_dc_power,DC power,W,W,1.0,0.0,inv1_dc_voltage*inv1_dc_current,avg,NaN,NaN,,inv1_dc_power__2717
19,1199,2722,inv2_dc_power,DC power,W,W,1.0,0.0,inv2_dc_voltage*inv2_dc_current,avg,NaN,NaN,,inv2_dc_power__2722
21,1199,2727,inv3_dc_power,DC power,W,W,1.0,0.0,inv3_dc_voltage*inv3_dc_current,avg,NaN,NaN,,inv3_dc_power__2727
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1728,50,765,dc_power_1,DC power,W,W,1.0,0.0,,avg,NaN,NaN,,dc_power_1__765
1729,50,766,dc_power_2,DC power,W,W,1.0,0.0,,avg,NaN,NaN,,dc_power_2__766
1747,51,772,dc_power,DC power,W,W,1.0,0.0,,avg,NaN,NaN,,dc_power__772
1749,51,786,dc_power_1,DC power,W,W,1.0,0.0,,avg,NaN,NaN,,dc_power_1__786


## Step 2: Get aggregate dc_power names

Rewrite given generalized code.

In [5]:
def find_aggregate_dc_power_names(
    print_messages: bool,
    sources_matter: bool = False,
    known_sources=('inverter',),  # meter not possible, that's AC
    known_sources_short=('inv',)
):
    dc_power_metrics = metrics_search_for_two_fragments_df(
        metrics_df, 'pow', 'dc', 'and'
    )
    var_name = 'dc_power'
    dc_agg_sensor_names = ['dc_power', 'dc_power_hW',
                           'dc_power_kW', 'dc_power_1_6',
                           'InvPDC_kW_Avg', 'dc_power_calc']
    return find_aggregate_variable_names_gen_mod(
        systems_cleaned=systems_cleaned,
        filtered_metrics_df=dc_power_metrics,
        var_name=var_name,
        agg_var_sensor_names = dc_agg_sensor_names,
        print_messages=print_messages,
        sources_matter=sources_matter,
        known_sources=known_sources,
        known_sources_short=known_sources_short
    )

In [6]:
dc_pow_agg_metrics, dc_power_agg_metadata = find_aggregate_dc_power_names(
    True, False
)

In [7]:
dc_pow_agg_metrics[1207]

[{'metric_id': np.int32(2622),
  'sensor_name': 'dc_power',
  'common_name': 'DC power',
  'units': 'W',
  'calc_details': '',
  'whole_or_part': 'whole'}]

In [28]:
dc_power_agg_metadata

,has_dc_power_aggregate
2,True
3,True
4,True
10,True
33,True
34,True
35,True
36,True
50,True
51,True


No need to add_aggs!

In [8]:
my_units = []
for id in dc_pow_agg_metrics.keys():
    for metric_dict in dc_pow_agg_metrics[id]:
        my_units.append(metric_dict['units'])
my_units = set(my_units)
my_units


{'W', 'kW'}

## Step 3: Get all DC power names

In [9]:
def find_all_dc_pow_metrics(
    print_messages: bool,
    sources_matter: bool = True,
    known_sources=('inverter',), 
    known_sources_short=('inv',)
):
    dc_pow_agg_metrics, dc_power_agg_metadata = find_aggregate_dc_power_names(
        print_messages=print_messages,
        sources_matter=sources_matter,
        known_sources=known_sources,
        known_sources_short=known_sources_short
    )
    dc_power_metrics = metrics_search_for_two_fragments_df(
        metrics_df, 'pow', 'dc', 'and'
    )
    # special filter -- dc_power_positive and dc_power_negative
    # not really subunits in the same way
    dc_power_cleaned_metrics = dc_power_metrics[
        ~dc_power_metrics['sensor_name'].str.contains('positive')
        & ~dc_power_metrics['sensor_name'].str.contains('negative')
    ]
    var_name = 'dc_power'
    dc_agg_sensor_names = ['dc_power', 'dc_power_hW',
                           'dc_power_kW', 'dc_power_1_6',
                           'InvPDC_kW_Avg', 'dc_power_calc']
    dc_pow_all_metrics, dc_pow_all_metadata = find_all_variable_names_gen_mod(
        var_aggs_dict=dc_pow_agg_metrics,
        var_aggs_metadata=dc_power_agg_metadata,
        filtered_metrics_df=dc_power_cleaned_metrics,
        var_name=var_name,
        agg_var_sensor_names=dc_agg_sensor_names,
        sources_matter=sources_matter,
        known_sources=known_sources,
        known_sources_short=known_sources_short
    )
    # problem with system 1207 -- bad units
    for system_id in dc_pow_agg_metrics.keys():
        for metric_dict in dc_pow_agg_metrics[system_id]:
            if metric_dict['units'] == '-':
                assert(system_id == 1207)
                metric_dict['units'] = 'W'  # from aggregate
    return (dc_pow_all_metrics, dc_pow_all_metadata)
    

In [10]:
dc_power_names, dc_power_subparts = find_all_dc_pow_metrics(
    False, False
)

In [52]:
dc_power_names

{2: [{'metric_id': np.int32(346),
   'sensor_name': 'dc_power',
   'common_name': 'DC power',
   'units': 'W',
   'calc_details': '',
   'whole_or_part': 'whole'}],
 3: [{'metric_id': np.int32(353),
   'sensor_name': 'dc_power',
   'common_name': 'DC power',
   'units': 'W',
   'calc_details': '',
   'whole_or_part': 'whole'}],
 1283: [{'metric_id': np.int32(1136),
   'sensor_name': 'dc_power',
   'common_name': 'DC power',
   'units': 'W',
   'calc_details': 'inv1_dc_power+inv2_dc_power',
   'whole_or_part': 'whole'},
  {'metric_id': np.int32(1134),
   'sensor_name': 'inv1_dc_power',
   'common_name': 'DC power',
   'units': 'W',
   'calc_details': 'inv1_dc_voltage*inv1_dc_current',
   'whole_or_part': 'part',
   'index': '1'},
  {'metric_id': np.int32(1135),
   'sensor_name': 'inv2_dc_power',
   'common_name': 'DC power',
   'units': 'W',
   'calc_details': 'inv2_dc_voltage*inv2_dc_current',
   'whole_or_part': 'part',
   'index': '2'}],
 1284: [{'metric_id': np.int32(943),
   'senso

Sanity checks

In [11]:
units_collection = []
for id in dc_power_names.keys():
    for metric_dict in dc_power_names[id]:
        units_collection.append(metric_dict['units'])
units_set = set(units_collection)
units_set
        

{'W', 'kW'}

In [12]:
part_metric_names = []
for system_id in dc_power_names.keys():
    for metric_dict in dc_power_names[system_id]:
        if metric_dict['whole_or_part'] == 'part':
            part_metric_names.append(metric_dict['sensor_name'])
part_metric_names.sort()
for metric_name in part_metric_names:
    print(metric_name)

ShuntPDC_kW_Avg_1
ShuntPDC_kW_Avg_1
ShuntPDC_kW_Avg_1
ShuntPDC_kW_Avg_2
ShuntPDC_kW_Avg_2
ShuntPDC_kW_Avg_2
ShuntPDC_kW_Avg_3
ShuntPDC_kW_Avg_3
ShuntPDC_kW_Avg_3
ShuntPDC_kW_Avg_4
ShuntPDC_kW_Avg_4
ShuntPDC_kW_Avg_4
ShuntPDC_kW_Avg_5
ShuntPDC_kW_Avg_5
ShuntPDC_kW_Avg_6
ShuntPDC_kW_Avg_6
ShuntPDC_kW_Avg_7
ShuntPDC_kW_Avg_7
dc_power_1
dc_power_1
dc_power_1_calc
dc_power_1_calc
dc_power_1_calc
dc_power_1_calc
dc_power_1_calc
dc_power_1_calc
dc_power_2
dc_power_2
dc_power_2_calc
dc_power_2_calc
dc_power_2_calc
dc_power_2_calc
dc_power_2_calc
dc_power_2_calc
dc_power_3_calc
dc_power_3_calc
dc_power_3_calc
dc_power_3_calc
dc_power_3_calc
dc_power_3_calc
dc_power_4_calc
dc_power_4_calc
dc_power_5_calc
dc_power_5_calc
dc_power_6_calc
dc_power_6_calc
dc_power_A1
dc_power_A2
dc_power_B1
dc_power_B2
dc_power_C1
dc_power_C2
inv1_dc_power
inv1_dc_power
inv1_dc_power
inv1_dc_power
inv1_dc_power
inv1_dc_power
inv1_dc_power
inv1_dc_power
inv1_dc_power
inv1_dc_power_hW
inv1_dc_power_hW
inv2_dc_power
in

Whole without part?

In [40]:
dc_power_names[1416]

[{'metric_id': np.int32(4742),
  'sensor_name': 'dc_power',
  'common_name': 'DC power',
  'units': 'W',
  'calc_details': '',
  'whole_or_part': 'whole'}]

Unfortunately, the unit-standardization requirement forces bespoke code here.
Also, sources do not matter here (avoiding major rewrites)

In [13]:
def dc_power_total_name(has_subparts: bool, unit: str = 'W'):
    '''Make the standardized variable name.'''
    total_name = 'dc_power'
    if has_subparts:
        total_name = total_name + '_total'
    total_name = total_name + '_' + unit
    return total_name


def dc_power_partial_name(ind: int, unit: str = 'W'):
    '''Make the standardized part-name.'''
    subpart_name = 'dc_power'
    subpart_name = subpart_name + f'_{ind}_{unit}'
    return subpart_name

In [28]:
def temp_agg_name():
    '''Provide the temporary manual-aggregate name just for reference later.'''
    return f'dc_power_artificial_sum'

### Original grab-all-data routine


In [14]:
def dc_power_dataframe_generator(
    system_id: int, 
    tall_or_wide: str,
    error_on_no_data: bool,
    size_standard: str,
):
    '''Make the (tall or wide) pandas DataFrame with all dc power data.'''
    dc_power_names, dc_power_metadata = find_all_dc_pow_metrics(
        print_messages=False,
        sources_matter=False
    )
    try:
        my_dc_power_names = dc_power_names[system_id]
        my_metadata = dc_power_metadata.loc[system_id, :]
    except KeyError:
        if error_on_no_data:
            raise ValueError(f'System {system_id} has no DC power data!')
        else:
            return None
    except BaseException as e:
        raise e
    metric_ids = []
    whole_metric_ids = []
    # grab all metric ids, putting the 'whole' category first
    for metric_data_dict in my_dc_power_names:
        if metric_data_dict['whole_or_part'] == 'whole':
            metric_ids.insert(0, metric_data_dict['metric_id'])
            whole_metric_ids.append(metric_data_dict['metric_id'])
        elif metric_data_dict['whole_or_part'] == 'part':
            metric_ids.append(metric_data_dict['metric_id'])
        else:
            raise ValueError('The "whole_or_part" result of sort_all_power_names_part_2()\n'
                             f'is not correct for system {system_id}.')
    # Load only these metrics from the system
    my_system_parquet_data_path = Path(f'../../../../data_ds_project/systems/parquet/{system_id}/')
    my_system_parquet_selection = pq.ParquetDataset(
        my_system_parquet_data_path, filters=[
            ('metric_id', 'in', metric_ids)
        ]
    )
    system_df = my_system_parquet_selection.read().to_pandas()
    # for reference, 4 columns (see
    # https://github.com/openEDI/documentation/blob/main/pvdaq.md#pvdaq_pvdata)
    # measured_on, utc_measured_on, metric_id, value)
    # standard cleaning
    system_df = system_df.drop_duplicates()
    # See if multiple values at a given time
    # if so, forced to replace value by mean value
    if any(system_df.duplicated(subset = ['measured_on', 'metric_id'])):
        system_df.loc[:, 'mean_value'] = system_df.groupby(
            ['measured_on', 'metric_id']
        )['value'].transform('mean')
        system_df = system_df.drop(columns='value')
        system_df = system_df.rename(columns={'mean_value':'value'})
        system_df.drop_duplicates()
    # if still duplicates, forced to drop utc_measured_on,
    # a frequent source of off-by-one-hour errors
    # (and points with the same 'measured_on' but different 'utc_measured_on'
    # have the same value, so it is likely that utc_measured_on is the problem)
    if any(system_df.duplicated(subset = ['measured_on', 'metric_id', 'value'])):
        system_df = system_df.drop(columns='utc_measured_on')
        system_df = system_df.drop_duplicates()
    # ready to widen the columns
    wide_df = system_df.pivot(
        index='measured_on',
        columns='metric_id',
        values='value'
    )
    # reset the metric_id name of the index of columns
    wide_df.columns.name = ''
    # reset the index
    wide_df = wide_df.reset_index()
    # before continuing, standardize the capitalization of the size term
    if size_standard.lower() == 'w':
        size_standard = 'W'
    elif size_standard.lower() == 'kw':
        size_standard = 'kW'
    else:
        raise ValueError('Only supports watts and kilowatts for now.')
    # standardize units -- probably only necessary for power and temperature
    # Irradiance is pretty clearly in W/m^2, current in A, voltage in V
    if size_standard == 'W':
        for metric_data_dict in my_dc_power_names:
            if metric_data_dict['units'].lower() == 'kw':
                wide_df.loc[:, metric_data_dict['metric_id']] = wide_df[metric_data_dict['metric_id']] * 1000
    elif size_standard == 'kW':
        for metric_data_dict in my_dc_power_names:
            if metric_data_dict['units'].lower() == 'w':
                wide_df.loc[:, metric_data_dict['metric_id']] = wide_df[metric_data_dict['metric_id']] / 1000
    else:
        raise ValueError('Only supports watts and kilowatts for now.')
    # push the 'whole' columns to the beginning of the pack
    # despite re-ordering earlier, can still be loaded in the wrong order.
    reordered_columns = ['measured_on'] + whole_metric_ids+ (wide_df.columns.drop(
        ['measured_on'] + whole_metric_ids
    ).tolist())
    wide_df = wide_df[reordered_columns]
    # rename columns
    renamer_dict = dict()
    for metric_data_dict in my_dc_power_names:
        if metric_data_dict['whole_or_part'] == 'whole':
            renamer_dict[metric_data_dict['metric_id']] = dc_power_total_name(
                has_subparts=my_metadata[metadata_part_name('dc_power')],
                unit=size_standard)
        elif metric_data_dict['whole_or_part'] == 'part':
            renamer_dict[metric_data_dict['metric_id']] = dc_power_partial_name(
                size_standard, metric_data_dict['index']
            )
        else:
            raise ValueError('The "whole_or_part" result of sort_all_power_names_part_2()\n'
                             f'is not correct for system {system_id}.')
    wide_df = wide_df.rename(columns=renamer_dict)
    # convert back to tall format if that is what we wanted
    if tall_or_wide == 'wide':
        our_df = wide_df
    elif tall_or_wide == 'tall':
        our_df = wide_df.melt(
            id_vars='measured_on',
            value_vars=list(renamer_dict.values()),
            var_name='metric_name',
            value_name='value'
        )
    else:
        raise ValueError('The term "tall_or_wide" must be "tall" or "wide.\n'
                         + f'Recieved {tall_or_wide}')
    return (our_df, renamer_dict)


### Test

In [15]:
(test_1207, names_1207) = dc_power_dataframe_generator(1207, 'wide', False, 'kW')

In [16]:
test_1207.head()

,measured_on,dc_power_total_kW,dc_power_kW_A1,dc_power_kW_A2,dc_power_kW_B1,dc_power_kW_B2,dc_power_kW_C1,dc_power_kW_C2
0,2011-04-22 15:40:00,3.0620,0.5329,0.5208,0.5217,0.5264,0.4953,0.4649
1,2011-04-22 15:50:00,5.0174,0.8660,0.8480,0.8590,0.8700,0.8180,0.7564
2,2011-04-22 16:00:00,3.8773,0.6714,0.6546,0.6653,0.6755,0.6355,0.5750
3,2011-04-22 16:10:00,2.9419,0.5111,0.4940,0.5041,0.5152,0.4817,0.4358
4,2011-04-22 16:20:00,2.2972,0.3989,0.3823,0.3936,0.4043,0.3730,0.3451


### Revision of the above (refrain from renaming until some cleaning done)

#### First, check which systems have "real" data

In [21]:
for system_id in tqdm(dc_power_names.keys()):
    (my_dc_power_names_trimmed, my_dc_power_meta_trimmed) = check_variable_data_exists_single_system(
        dc_power_names,
        dc_power_subparts,
        '../../../../data_ds_project/systems/parquet/',
        system_id,
        'dc_power',
        sources_matter=False,
    )
    if my_dc_power_names_trimmed is None:
        print(f'System {system_id} has no dc power data!')
    elif len(dc_power_names[system_id]) > len(my_dc_power_names_trimmed):
        print(f'System {system_id} has lost some dc power data.')

 23%|██▎       | 11/48 [05:20<12:35, 20.42s/it]

System 1422 has no dc power data!


 25%|██▌       | 12/48 [05:52<14:18, 23.84s/it]

System 1423 has no dc power data!


 29%|██▉       | 14/48 [06:11<09:10, 16.19s/it]

System 1429 has no dc power data!


100%|██████████| 48/48 [30:12<00:00, 37.77s/it]


Again, 1422, 1423, 1429 appear to have missing DC power points -- it appears that they have no power values.

In [29]:
def dc_power_gather_data(
    all_dc_pow_metrics,
    all_dc_pow_metadata: pd.DataFrame,
    system_id: int,
    error_on_no_data: bool,
    add_aggs: bool,
    known_sources = ('inverter', 'meter'),
    known_sources_short = ('inv', 'met'),
):
    '''Gather all dc power-data per-system,
    given a list of aggregate sensor names.
    
    Parameters
    ----------
    all_dc_pow_metrics: dict[list[dict]]
        first result of find_all_dc_power_metrics(*args)
    all_dc_pow_metadata
        second result of find_all_dc_power_metrics(*args)
    system_id: int
        Index of system in systems_cleaned and metric_df
    error_in_no_data: bool
        If True, return an error if the system_id has no ac-power data.
        If False, return None if the system-system_id has no ac-power data.
    add_aggs: bool
        If True, and there are parts without a corresponding aggregate,
            add the aggregate, according to agg_type.
        If False, do nothing.
    known_sources: iterable of strings
        Full names of the known source types.
    known_sources_short: iterable of strings
        fragments of the known source names suitable for searching

    Returns
    --------
    A pandas DataFrame object with the desired data.
    '''
    known_sources, known_sources_short = sources_checker(
        known_sources, known_sources_short
    )
    # specialize to current ID number
    try:
        my_dc_power_names = deepcopy(all_dc_pow_metrics[system_id])
        my_dc_power_metadata = all_dc_pow_metadata.loc[system_id, :]
    except KeyError:
        if error_on_no_data:
            raise ValueError(f'System {system_id} has no AC Power data!')
        else:
            return None
    except BaseException as e:
        raise e
    # for known problem-cases, clean the data of spurious variables
    if system_id in [1422, 1423, 1429]:
        (my_dc_power_names, my_dc_power_metadata) = check_variable_data_exists_single_system(
            var_total_dict=all_dc_pow_metrics,
            var_total_metadata_df=all_dc_pow_metadata,
            path_to_raw_data_dir='../../../../data_ds_project/systems/parquet/',
            system_id=system_id,
            var_name='dc_power',
            sources_matter=False,
            known_sources=known_sources,
            known_sources_short=known_sources_short
        )
        assert(my_dc_power_names is None)
        return (None, None, None)
    # grab some metadata, quickly
    metric_ids = []
    whole_metric_ids = []
    # grab all metric ids, putting the 'whole' category first
    for metric_data_dict in my_dc_power_names:
        if metric_data_dict['whole_or_part'] == 'whole':
            metric_ids.insert(0, metric_data_dict['metric_id'])
            whole_metric_ids.append(metric_data_dict['metric_id'])
        elif metric_data_dict['whole_or_part'] == 'part':
            metric_ids.append(metric_data_dict['metric_id'])
        else:
            raise ValueError('The "whole_or_part" result of find_all_ac_power_names()\n'
                             f'is not correct for system {system_id}.')
    # Load only these metrics from the system
    my_system_parquet_data_path = Path(f'../../../../data_ds_project/systems/parquet/{system_id}/')
    my_system_parquet_selection = pq.ParquetDataset(
        my_system_parquet_data_path, filters=[
            ('metric_id', 'in', metric_ids)
        ]
    )
    system_df = my_system_parquet_selection.read().to_pandas()
    # for reference, 4 columns (see
    # https://github.com/openEDI/documentation/blob/main/pvdaq.md#pvdaq_pvdata)
    # measured_on, utc_measured_on, metric_id, value)
    # standard cleaning
    # do we actually have all of the data
    if len(set(system_df['metric_id'])) != len(metric_ids):
        missing_ids = set(metric_ids).difference(set(system_df['metric_id']))
        print(f'Missing ids: {missing_ids}')
        raise ValueError('metrics not found!')


    system_df = system_df.drop_duplicates()
    # See if multiple values at a given time
    # if so, forced to replace value by mean value
    if any(system_df.duplicated(subset = ['measured_on', 'metric_id'])):
        system_df.loc[:, 'mean_value'] = system_df.groupby(
            ['measured_on', 'metric_id']
        )['value'].transform('mean')
        system_df = system_df.drop(columns='value')
        system_df = system_df.rename(columns={'mean_value':'value'})
        system_df.drop_duplicates()
    # if still duplicates, forced to drop utc_measured_on,
    # a frequent source of off-by-one-hour errors
    # (and points with the same 'measured_on' but different 'utc_measured_on'
    # have the same value, so it is likely that utc_measured_on is the problem)
    if any(system_df.duplicated(subset = ['measured_on', 'metric_id', 'value'])):
        system_df = system_df.drop(columns='utc_measured_on')
        system_df = system_df.drop_duplicates()
    # ready to widen the columns
    wide_df = system_df.pivot(
        index='measured_on',
        columns='metric_id',
        values='value'
    )
    # reset the metric_id name of the index of columns
    wide_df.columns.name = ''
    # reset the index
    wide_df = wide_df.reset_index()
    # Some systems have part-data and not aggregate data;  
    # amend this mistake.
    if add_aggs:
        for source_type in known_sources:
            if (my_dc_power_metadata[metadata_part_name('dc_power')])\
              and (not my_dc_power_metadata[metadata_part_name('dc_power')]):
                source_type_total_name = temp_agg_name()
                # as agg_type == 'sum' in gen_variable_standardizer
                wide_df[source_type_total_name] = wide_df.apply(
                    lambda row: np.sum(
                        [row[j] for j in metric_ids]
                    ), axis=1
                )
                sensor_names_summed = []
                units_group = []
                for metric_dict in my_dc_power_names:
                    if metric_dict['metric_id'] in metric_ids:
                        sensor_names_summed.append(metric_dict['sensor_name'])
                        units_group.append(metric_dict['units'])
                calc_type = sensor_names_summed[0]
                for j in range(1, len(sensor_names_summed)):
                    calc_type = f'{calc_type} + {sensor_names_summed[j]}'
                units_group = set(units_group)
                if len(units_group) == 1:
                    # grab the singleton as our unit
                    # see https://stackoverflow.com/questions/1619514/how-to-extract-the-member-from-single-member-set-in-python
                    # for more info on this
                    (my_unit, ) = units_group
                else:
                    raise RuntimeError('Multiple units in the subparts summed!  No good!')
                whole_metric_ids.append(source_type_total_name)
                # adjoin new variables to our metadata lists as well
                my_dc_power_metadata[metadata_agg_name('dc_power')] = True
                my_dc_power_names.append(
                    {
                        'sensor_name': source_type_total_name,
                        'units': my_unit,
                        'calc_type': calc_type,
                        'common_name': 'AC power',
                        'metric_id': 'N/A',
                        'whole_or_part': 'whole',
                    }
                )

    # preserve 'whole' columns
    whole_columns = ['measured_on',] + whole_metric_ids 
    wide_df = wide_df[whole_columns]
    renamer_dict = dict()
    for metric_data_dict in my_dc_power_names:
        renamer_dict[metric_data_dict['metric_id']] = metric_data_dict['sensor_name']
    wide_df = wide_df.rename(columns=renamer_dict)
    
    return(my_dc_power_names, my_dc_power_metadata, wide_df)
    

In [30]:
dc_power_gather_data(
    dc_power_names,
    dc_power_subparts,
    2,
    True,
    True,
)

([{'metric_id': np.int32(346),
   'sensor_name': 'dc_power',
   'common_name': 'DC power',
   'units': 'W',
   'calc_details': '',
   'whole_or_part': 'whole'}],
 has_dc_power_aggregate      True
 has_dc_power_subsystems    False
 Name: 2, dtype: bool,
                 measured_on  dc_power
 0       2010-01-21 11:02:00  13.31000
 1       2010-01-21 11:03:00  13.06000
 2       2010-01-21 11:04:00  13.07000
 3       2010-01-21 11:05:00  13.06000
 4       2010-01-21 11:06:00  12.48000
 ...                     ...       ...
 1910144 2020-01-13 16:25:00  28.74832
 1910145 2020-01-13 16:30:00  26.35972
 1910146 2020-01-13 16:35:00  21.83201
 1910147 2020-01-13 16:40:00  10.90820
 1910148 2020-01-13 16:45:00  11.79171
 
 [1910149 rows x 2 columns])

In [31]:
dc_power_gather_data(
    dc_power_names,
    dc_power_subparts,
    1422,
    True,
    True,
)

(None, None, None)

In [32]:
for system_id in dc_power_names.keys():
    dc_power_gather_data(dc_power_names, dc_power_subparts, system_id, True, True)

KeyboardInterrupt: 

## Testing Ground

Verifying 1422, 1423, 1429

In [22]:
system_id = 1422
relevant_rows_metrics = metrics_df[metrics_df['system_id'] == system_id]
metrics_search_for_two_fragments_df(relevant_rows_metrics, 'pow', 'dc', 'and')

,system_id,metric_id,sensor_name,common_name,raw_units,units,calc_scale,calc_offset,calc_details,aggregation_type,source_type,source_id,comments,standard_name
1177,1422,5376,dc_power,DC power,W,W,1.0,0.0,,avg,NaN,NaN,,dc_power__5376


In [23]:
path_1422 = Path(f'../../../../data_ds_project/systems/parquet/1422/')
pq_1422  = pq.ParquetDataset(
    path_1422, filters=[
            ('metric_id', 'in', [5376])
    ]
)
df_1422 = pq_1422.read().to_pandas()
df_1422.tail()

,measured_on,utc_measured_on,metric_id,value


In [24]:
system_id = 1423
relevant_rows_metrics = metrics_df[metrics_df['system_id'] == system_id]
metrics_search_for_two_fragments_df(relevant_rows_metrics, 'pow', 'dc', 'and')

,system_id,metric_id,sensor_name,common_name,raw_units,units,calc_scale,calc_offset,calc_details,aggregation_type,source_type,source_id,comments,standard_name
1203,1423,5378,dc_power,DC power,W,W,1.0,0.0,,avg,NaN,NaN,,dc_power__5378


In [25]:
path_1423 = Path(f'../../../../data_ds_project/systems/parquet/1423/')
pq_1423  = pq.ParquetDataset(
    path_1423, filters=[
            ('metric_id', 'in', [5378])
    ]
)
df_1423 = pq_1423.read().to_pandas()
df_1423.tail()

,measured_on,utc_measured_on,metric_id,value


In [26]:
system_id = 1429
relevant_rows_metrics = metrics_df[metrics_df['system_id'] == system_id]
metrics_search_for_two_fragments_df(relevant_rows_metrics, 'pow', 'dc', 'and')

,system_id,metric_id,sensor_name,common_name,raw_units,units,calc_scale,calc_offset,calc_details,aggregation_type,source_type,source_id,comments,standard_name
1229,1429,5380,dc_power,DC power,W,W,1.0,0.0,,avg,NaN,NaN,,dc_power__5380


In [27]:
path_1429 = Path(f'../../../../data_ds_project/systems/parquet/1429/')
pq_1429  = pq.ParquetDataset(
    path_1429, filters=[
            ('metric_id', 'in', [5380])
    ]
)
df_1429 = pq_1429.read().to_pandas()
df_1429.tail()

,measured_on,utc_measured_on,metric_id,value


Affirmed -- systems 1422, 1423, 1429 have no dc power data.